In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Rachita Sharma and Pavani Priyanka Kotturi Final Project CS5100

In [2]:

data = pd.read_excel(r"data/UWSM_Data.xlsx")

data.head()

,Unique ID,Survey Type,Start time,What are the biggest challenges facing households in your community today?,Do you currently live or work in Southern Maine? (Can select more than one),"If an unexpected expense of $400 came up, would you be able to pay it?",Are you part of a community whose voice you feel is not being heard?,"What community are you referring to? Please name the group, town, or city.",What is your age group?,How do you describe your race or ethnicity? (Can select more than one),What is your primary relationship with United Way of Southern Maine (UWSM)?,Want to get involved?,How would you consider getting involved locally to be part of the solution? (Select all that apply),Which bills or everyday expenses are the hardest for your household to afford?,What is your zip code?,Are you currently employed?,What type of employment?,What’s getting in the way of being able to cover your bills or living expenses? (Select all that apply),What supports would be most helpful for you?
0,CASH1,CASH,45889 days 16:18:22,NaN,Other Maine County,Yes,Yes,65 plus We often have reduced income but with ...,NaN,NaN,NaN,NaN,I am not interested or able to at this time;,"Transportation (e.g., car payments, gas, car r...",4103.0,No,NaN,I’m 76 years old;,Increased income;
1,CASH2,CASH,45889 days 16:18:49,NaN,Other Maine County,"Maybe, with difficulty",Maybe,the Village of Gray,NaN,NaN,NaN,NaN,I am not interested or able to at this time;,"Utilities (e.g., electricity, water, gas);Insu...",4039.0,No,NaN,I am retired and didn't plan ahead.;,Help affording healthcare or mental health sup...
2,CASH3,CASH,45889 days 16:43:22,NaN,Other Maine County,No,Yes,South Portland,NaN,NaN,NaN,NaN,"I’m not sure yet, but I’d like to learn more;","Housing costs (e.g., rent, mortgage);Utilities...",4106.0,No,NaN,Poor health;,Rental assistance;Help with transportation (ga...
3,CASH4,CASH,45889 days 16:57:55,NaN,Other Maine County,"Maybe, with difficulty",No,NaN,NaN,NaN,NaN,NaN,Language volunteer ;,Medical bills/Healthcare Costs;Transportation ...,4103.0,Yes,Full-time hours (40 or more) working for one e...,I’m not sure what opportunities are available;,Help affording healthcare or mental health sup...
4,CASH5,CASH,45889 days 18:58:47,NaN,Other Maine County,No,Yes,People living with Long Covid,NaN,NaN,NaN,NaN,Preparing to share my story with the Maine st...,"Housing costs (e.g., rent, mortgage);Medical b...",4103.0,Yes,Freelance/contract,I have health issues or a disability;,Rental assistance;Help affording healthcare or...


In [3]:
##data.info()
##data.describe()
data.columns

Index(['Unique ID', 'Survey Type', 'Start time',
       'What are the biggest challenges facing households in your community today?',
       'Do you currently live or work in Southern Maine? (Can select more than one)',
       'If an unexpected expense of $400 came up, would you be able to pay it?',
       'Are you part of a community whose voice you feel is not being heard?',
       'What community are you referring to?  Please name the group, town, or city. ',
       'What is your age group?',
       'How do you describe your race or ethnicity? (Can select more than one)',
       'What is your primary relationship with United Way of Southern Maine (UWSM)?',
       'Want to get involved?  ',
       'How would you consider getting involved locally to be part of the solution? (Select all that apply)',
       'Which bills or everyday expenses are the hardest for your household to afford?',
       'What is your zip code?', 'Are you currently employed?',
       'What type of employment?',


In [4]:
data = data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [5]:
for col in data.columns: # have to do this otherwise the code breaks
    if "community are you referring to" in col:
        data = data.rename(columns={col: "community_group"})
        break
for col in data.columns: # have to do this otherwise the code breaks
    if "What’s getting in the way of being able to cover your bills or living expenses? (Select all that apply)" in col:
        data = data.rename(columns={col: "expense_barriers"})
        break

In [6]:
def map_semicolon_col(series, mapping):
    """Map each ;-separated item using a dict, drop unmapped items, rejoin."""
    def _map_row(val):
        if not isinstance(val, str) or not val.strip():
            return pd.NA
        coded = [mapping.get(i.strip()) for i in val.split(";") if i.strip()]
        coded = [c for c in coded if c is not None]
        return ";".join(coded) if coded else pd.NA
    return series.apply(_map_row)

data = data.rename(columns={
    "Unique ID": "id",
    "Survey Type": "survey_type",
    "Start time": "start_time",
    "What are the biggest challenges facing households in your community today?": "challenges",
    "Do you currently live or work in Southern Maine? (Can select more than one)": "County",
    "If an unexpected expense of $400 came up, would you be able to pay it?": "expense_400",
    "Are you part of a community whose voice you feel is not being heard?": "unheard",
    "What is your age group?": "age_group",
    "How do you describe your race or ethnicity? (Can select more than one)": "race",
    "What is your primary relationship with United Way of Southern Maine (UWSM)?": "uw_connection",
    "Want to get involved?": "want_involved",
    "How would you consider getting involved locally to be part of the solution? (Select all that apply)": "engagement",
    "Which bills or everyday expenses are the hardest for your household to afford?": "hardest_expenses",
    "What is your zip code?": "zip",
    "Are you currently employed?": "employment_status",
    "What type of employment?": "employment_type",
    "What supports would be most helpful for you?": "supports",
})
data.head()

,id,survey_type,start_time,challenges,County,expense_400,unheard,community_group,age_group,race,uw_connection,Want to get involved?,engagement,hardest_expenses,zip,employment_status,employment_type,expense_barriers,supports
0,CASH1,CASH,45889 days 16:18:22,NaN,Other Maine County,Yes,Yes,65 plus We often have reduced income but with ...,NaN,NaN,NaN,NaN,I am not interested or able to at this time;,"Transportation (e.g., car payments, gas, car r...",4103.0,No,NaN,I’m 76 years old;,Increased income;
1,CASH2,CASH,45889 days 16:18:49,NaN,Other Maine County,"Maybe, with difficulty",Maybe,the Village of Gray,NaN,NaN,NaN,NaN,I am not interested or able to at this time;,"Utilities (e.g., electricity, water, gas);Insu...",4039.0,No,NaN,I am retired and didn't plan ahead.;,Help affording healthcare or mental health sup...
2,CASH3,CASH,45889 days 16:43:22,NaN,Other Maine County,No,Yes,South Portland,NaN,NaN,NaN,NaN,"I’m not sure yet, but I’d like to learn more;","Housing costs (e.g., rent, mortgage);Utilities...",4106.0,No,NaN,Poor health;,Rental assistance;Help with transportation (ga...
3,CASH4,CASH,45889 days 16:57:55,NaN,Other Maine County,"Maybe, with difficulty",No,NaN,NaN,NaN,NaN,NaN,Language volunteer ;,Medical bills/Healthcare Costs;Transportation ...,4103.0,Yes,Full-time hours (40 or more) working for one e...,I’m not sure what opportunities are available;,Help affording healthcare or mental health sup...
4,CASH5,CASH,45889 days 18:58:47,NaN,Other Maine County,No,Yes,People living with Long Covid,NaN,NaN,NaN,NaN,Preparing to share my story with the Maine sta...,"Housing costs (e.g., rent, mortgage);Medical b...",4103.0,Yes,Freelance/contract,I have health issues or a disability;,Rental assistance;Help affording healthcare or...


In [7]:
challenge_map = {
    # Standard checkbox options
    "Paying for housing (rent, utilities, mortgage, taxes)": "Housing Cost",
    "Covering basic expenses (groceries, gas, emergencies)": "Cost of Basics",
    "Getting enough food": "Food Access",
    "Finding affordable healthcare": "Healthcare Cost",
    "Mental health challenges (like anxiety, depression)": "Mental Health",
    "Behavioral health challenges (like substance use or addiction)": "Behavioral Health",
    "Child care that is affordable and available": "Affordable Child Care Access",
    "Caring for aging parents or relatives": "Affordable Elder Care Access",
    "Getting to work, school, or appointments (transportation)": "Transportation",
    "Jobs with low pay or no benefits": "Low Wages",
    "Lack of stable jobs with career growth": "Job Instability",
    "Feeling isolated or not connected to community": "Social Isolation",
    "Access to college or training after high school": "Education Access",
    "Impact of flooding and other extreme weather": "Climate Impacts",
    # Write-in codes (per codebook)
    "Finding affordable housing": "Affordable Housing Availability",
    "Affordable Housing": "Affordable Housing Availability",
    "affordable housing": "Affordable Housing Availability",
    "FINDING HOUSING": "Affordable Housing Availability",
    "Housing": "Affordable Housing Availability",
    "Housing shortage and affordability (rent, utilities, mortgage, taxes)": "Affordable Housing Availability",
    "Lack of affordable housing before even signing a rental or purchase agreement": "Affordable Housing Availability",
    "finding affordable & available housing": "Affordable Housing Availability",
    "finding housing in their budget within geography": "Affordable Housing Availability",
    "Affordable house for low income workers.  And healthcare providers in insurers network": "Affordable Housing Availability",
    "Language skills": "Language Access",
    "language access": "Language Access",
    "High taxes": "Rising Taxes",
    "Political division anger and outrage created by entertainment \"news\" and social media": "Political Division",
    "too much political division, more unity encouragement is needed": "Political Division",
    "Dealing with current government administration": "Political Division",
    "Dismantling of our democracy": "Political Division",
    "Liberals conspiring to block federal government efforts": "Political Division",
    "ICE": "Immigration Enforcement",
    "ICE influencing daily things": "Immigration Enforcement",
    "impact of ICE": "Immigration Enforcement",
    "We cannot sustain the number of people on welfare sucking the system dry!": "Welfare Dependence",
    "Lack of in home support": "Lack of in Home Support",
    "Gas mileage": "Gas Costs",
    "energy costs": "Gas Costs",
    "Rent keeps going up!": "Rent Increases",
    "Checked all": "Other",
    "Caring for A Behavior/Developement impaired adult child": "Other",
    "Lack of places where people can join together for free or low-cost concerts, activities, etc.": "Other",
    "Mattresses and furniture delivery for people who have experienced homelessness and are moving into their first apartment. Bus passes and motor vehicle repair is also needed desperately.": "Other",
    "getting dental, vision insurance and vet care for my dog": "Other",
    "good paying job is not enough to pay the bills": "Other",
    "particularly mortgage and emergencies": "Other",
    # Near-duplicates of standard options
    "Food insecurity (hard to afford or access food)": "Food Access",
    "The cost of food is way too high": "Food Access",
    "Rising cost of\xa0basics\xa0(like food, utilities, essentials)": "Cost of Basics",
    "Mental health challenges": "Mental Health",
    "Limited access to affordable\xa0healthcare": "Healthcare Cost",
    "Limited access to, and affordability of,\xa0child care": "Affordable Child Care Access",
    "Isolation\xa0/ disconnected community": "Social Isolation",
}


In [8]:

alice_map = {
    "Maybe, with difficulty": "Below ALICE",
    "No": "Below ALICE",
    "Yes": "Above ALICE"
}
 
data["ALICE_status"] = data["expense_400"].map(alice_map)

In [9]:
age_map = {
    "Under 18": "Under 24",
    "18 - 24": "Under 24",
    "25 - 44": "25-64",
    "45 - 64": "25-64",
    "65 - 75": "Over 65+",
    "More than 75": "Over 65+",
    "Prefer not to answer": None
}
 
data["Age"] = data["age_group"].str.strip().map(age_map)

In [10]:
data['zip'] = data['zip'].astype(str).str.zfill(5)
 
zip_to_county = {
    "00402": "Outside Southern Maine",
    "03901": "York",
    "04005": "York",
    "04009": "Cumberland",
    "04011": "Cumberland",
    "04021": "Cumberland",
    "04032": "Cumberland",
    "04038": "Cumberland",
    "04039": "Cumberland",
    "04042": "York",
    "04049": "York",
    "04062": "Cumberland",
    "04072": "York",
    "04074": "Cumberland",
    "04083": "York",
    "04085": "Cumberland",
    "04261": "Outside Southern Maine",
    "04401": "Outside Southern Maine",
    "04438": "Outside Southern Maine",
    "94074": "Outside Southern Maine"
}
 
county_to_region = {
    "Androscoggin": "Outside Southern Maine",
    "Cumberland": "Cumberland",
    "Cumberland & York": "Cumberland & York",
    "York": "York",
    "Franklin County": "Outside Southern Maine",
    "Kennebec": "Outside Southern Maine",
    "Knox": "Outside Southern Maine",
    "Lincoln": "Outside Southern Maine",
    "Other": "Outside Southern Maine",
    "Oxford": "Outside Southern Maine",
    "Penobscot": "Outside Southern Maine",
    "Sagadahoc": "Outside Southern Maine",
    "Somerset": "Outside Southern Maine"
}
 
data['zip_county'] = data['zip'].map(zip_to_county)
data['County'] = data['County'].fillna(data['zip_county'])
data['region'] = data['County'].map(county_to_region)
 
data = data[data['region'].isin(['Cumberland', 'York', 'Cumberland & York'])]

In [11]:

unheard_map = {
    "Maybe": "Unheard",
    "No": "Heard",
    "Yes": "Unheard"
}
 
data["Community_Voice"] = data["unheard"].map(unheard_map)

In [12]:
community_map = {
    "African American": "African American",
    "Asian": "Asian",
    "Everyone": "Other",
    "Foster kids": "Foster Kids",
    "Hispanic": "Hispanic",
    "Immigrant": "Immigrant",
    "Indigenous": "Indigenous",
    "LGBTQ+": "LGBTQ+",
    "Low Income": "ALICE",
    "Noble School": "Youth (under 18)",
    "Older adults": "Older Adults (+65)",
    "Parents": "Parents",
    "Recovery community": "Recovery Community",
    "Single mothers": "Single Mothers",
    "Town": "Town of Residence",
    "White": "White",
    "Women": "Women",
    "Youth": "Youth (under 18)"
}
 
data['community_clean'] = data['community_group'].map(community_map)
data['community_clean'] = data['community_clean'].fillna("Other")
 
community_map_broad = {
    "African American": "BIPOC",
    "Asian": "BIPOC",
    "Everyone": "Other",
    "Foster kids": "Other",
    "Hispanic": "BIPOC",
    "Immigrant": "Immigrant",
    "Indigenous": "BIPOC",
    "LGBTQ+": "LGBTQ+",
    "Low Income": "ALICE",
    "Noble School": "Youth (under 18)",
    "Older adults": "Older Adults (+65)",
    "Parents": "Parents",
    "Recovery community": "Recovery Community",
    "Single mothers": "Single Mothers",
    "Town": "Town of Residence",
    "White": "White",
    "Women": "Women",
    "Youth": "Youth (under 18)"
}
 
data['community_clean_broad'] = data['community_group'].map(community_map_broad)

In [13]:
race_map = {
    "American Indian or Alaska Native": "BIPOC",
    "Asian": "BIPOC",
    "Black or African American": "BIPOC",
    "Hispanic, Latino, or Latina/o/x": "BIPOC",
    "Hispanic": "BIPOC",
    "More than one": "More than one",
    "Native Hawaiian or Pacific Islander": "BIPOC",
    "Native Hawaiian or Pacific Islander; White": "More than one",
    "Other": "Other",
    "Prefer not to answer": "Prefer not to answer",
    "White": "White"
}
 

In [14]:
uw_connection_map = {
    "Formerly Involved": "Formerly Connected",
    "I am closely connected (I am a UWSM volunteer, staff, or partner organization)": "Closely Connected",
    "I am moderately connected (For example, I occasionally volunteer and/or donate to UWSM)": "Moderately Connected",
    "I am not connected with United Way": "Not Connected"
}
 
data['UW_connection'] = data["uw_connection"].map(uw_connection_map)
data['UW_connection'] = data['UW_connection'].fillna("Not Connected")

In [15]:
engagement_map = {
    "Donating money to United Way of Southern Maine": "Donate",
    "Donating to United Way of Southern Maine": "Donate",
    "Helping connect people to services or resources": "Connect",
    "Speaking up or helping spread the word about important issues": "Speak Up",
    "Speaking up or helping spread the word about important policy issues": "Speak Up",
    "Joining a group that solves community issues": "Join Group",
    "Sharing my story or experiences to help others understand": "Share Story",
    "Volunteering my time or skills in local programs": "Volunteer"
}
 


In [16]:
employment_map = {
    "32 hours": "Full-Time - 1 Job",
    "a mixture of part time and self-employed": "Other",
    "Freelance/contract": "Other",
    "Full-time hours (40 or more) working for more than one employer": "Full-Time - Multiple Jobs",
    "Full-time hours (40 or more) working for one employer": "Full-Time - 1 Job",
    "I am election clerk working 3 times a year": "Other",
    "Part-time hours (29 or less)": "Part-Time",
    "Per diem": "Per Diem",
    "Self employed": "Self Employed"
}
 
data['Employment Type'] = data["employment_type"].map(employment_map)

In [17]:
expense_barrier_map = {
    # Standard options
    "I don't have reliable transportation": "Lack of Reliable Transportation",
    "I'm caring for a family member": "Family Care Responsibilities",
    "I have health issues or a disability": "Health Issues or Disability",
    "I don't have time to go back to school or get more training": "Lack of Time for Training/Education",
    "I don't have the skills or training needed for a better-paying job": "Lack of Skills or Training",
    "I've been turned away from jobs because of my background": "Turned Away",
    "I don't have affordable child care": "Lack of Affordable Child Care",
    "I'm not sure what opportunities are available": "Unaware of or Limited Job Opportunities",
    # Write-in codes (per codebook)
    "Age-Related Employment Barriers": "Age-Related Employment Barriers",
    "Insufficient Income / Fixed Income": "Insufficient Income / Fixed Income",
    "High Cost of Living": "High Cost of Living",
    "Retirement Challenges": "Retirement Challenges",
    "Debt / Cash Flow Issues": "Debt / Cash Flow Issues",
    "Financially Stable / Doing Well": "Financially Stable / Doing Well"
}
 


In [18]:

support_map = {
    # Standard options
    "Help with food or groceries": "Food",
    "Help affording healthcare or mental health support": "Health/Mental Health",
    "Help with transportation (gas, car repair, bus passes, etc.)": "Transportation",
    "Help with job training or education": "Job Training/Education",
    "Help managing debt or improving credit": "Debt/Credit",
    "Employment navigation": "Employment",
    "Rental assistance": "Rental Assistance",
    "Help navigating programs like SNAP, TANF": "SNAP/TANF Navigation",
    # Write-in codes (per codebook)
    "Increased Income": "Increased Income",
    "Taxes": "Taxes",
    "Home Ownership": "Home Ownership"
}
 
 

In [19]:
data["challenge_codes"]      = map_semicolon_col(data["challenges"], challenge_map)
data["race_codes"]           = map_semicolon_col(data["race"], race_map)
data["engagement_codes"]     = map_semicolon_col(data["engagement"], engagement_map)
data["expense_barrier_codes"] = map_semicolon_col(data["expense_barriers"], expense_barrier_map)
data["support_codes"]        = map_semicolon_col(data["supports"], support_map)
 
respondents = data.copy()


In [20]:
respondents.to_excel("Respondents_dataset.xlsx", index=False)


In [21]:
challenge_long = respondents.copy()
challenge_long["challenges"] = challenge_long["challenges"].str.split(";")
challenge_long = challenge_long.explode("challenges")
challenge_long["challenges"] = challenge_long["challenges"].str.strip()
challenge_long["challenge_code"] = challenge_long["challenges"].map(challenge_map)

race_long = respondents.copy()
race_long["race"] = race_long["race"].str.split(";")
race_long = race_long.explode("race")
race_long["race"] = race_long["race"].str.strip()
race_long["race_code"] = race_long["race"].map(race_map)

engagement_long = respondents.copy()
engagement_long["engagement"] = engagement_long["engagement"].str.split(";")
engagement_long = engagement_long.explode("engagement")
engagement_long["engagement"] = engagement_long["engagement"].str.strip()
engagement_long["Engagement"] = engagement_long["engagement"].map(engagement_map)
 


expense_barriers_long = respondents.copy()
expense_barriers_long["expense_barriers"] = expense_barriers_long["expense_barriers"].str.split(";")
expense_barriers_long = expense_barriers_long.explode("expense_barriers")
expense_barriers_long["expense_barriers"] = expense_barriers_long["expense_barriers"].str.strip()
expense_barriers_long["Expense_barriers"] = expense_barriers_long["expense_barriers"].map(expense_barrier_map)


supports_long = respondents.copy()
supports_long["supports"] = supports_long["supports"].str.split(";")
supports_long = supports_long.explode("supports")
supports_long["supports"] = supports_long["supports"].str.strip()
supports_long["Supports"] = supports_long["supports"].map(support_map)


In [24]:
with pd.ExcelWriter("UWSM_Analysis_Datasets.xlsx") as writer:
    respondents.to_excel(writer, sheet_name="Respondents", index=False)
    challenge_long.to_excel(writer, sheet_name="Challenge_Long", index=False)
    race_long.to_excel(writer, sheet_name="Race_Long", index=False)
    engagement_long.to_excel(writer, sheet_name="Engagement_Long", index=False)
    expense_barriers_long.to_excel(writer, sheet_name="Expense_Barriers_Long", index=False)
    supports_long.to_excel(writer, sheet_name="Supports_Long", index=False)